<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/04-intro_to_numpy/04-intro_to_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Introduction to NumPy (Numeric Python)

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0)
- Gorman KB, Williams TD, Fraser WR (2014). *Ecological Sexual Dimorphism and Environmental Variability within a Community of Antarctic Penguins (Genus Pygoscelis).* PLoS ONE 9(3): e90081

*Last major revision: 2026-07-30*

## Key Concepts

- Explain why plain lists fail for element-wise math
- Create NumPy arrays and use element-wise operations and broadcasting
- Filter data with boolean masks
- Work with 2D arrays: shape, indexing, slicing, transpose, and the `axis` argument
- Compute summary statistics and a correlation, and recognize **Simpson's paradox** when a pooled correlation reverses within groups

**A note on the code cells.** Cells marked `# RUN-TOGETHER` are complete and will run as written. Cells marked `# FILL-IN` contain blanks written as `____` and will raise a `SyntaxError` until every blank is replaced.

---
## 1. Why Arrays?
---
### 1.1 The issue with lists

In data science you will often want to carry out an operation over an entire collection of values, and you want it to be fast.

With lists, this is a problem.

In [3]:
# RUN-TOGETHER
# This cell is SUPPOSED to fail. Read the error message.
# Suppose we have height and weight data for a collection of people.
# Our goal is to calculate BMI for each person: BMI = weight / height^2

height = [1.73, 1.72, 1.61, 1.89, 1.79]
weight = [65.4, 59.2, 63.6, 88.4, 68.7]

bmi = weight / height ** 2

TypeError: unsupported operand type(s) for ** or pow(): 'list' and 'int'

For a **list**, `**` and `/` are not defined element by element. There is no way to say "divide each weight by each height squared" without writing a loop. We want something that just does the math. That is NumPy.

### 1.2 NumPy to the rescue

**Note:** you may need to install NumPy on your own system before using the package. In Google Colab it is already available.

NumPy provides an alternative to lists: arrays.

**Definition.** A NumPy *array* (`ndarray`) is an ordered collection of values, like a list, but with two superpowers: (1) math happens **element-wise** across the whole array, and (2) it is fast, because the looping runs in compiled C rather than in Python.

One rule: a NumPy array can contain only **one** type of data.

- If you create a "mixed" array, Python converts the values to a more general type (such as `str`)
- NumPy arrays are another type of object, like `str`, `float`, and `list`

In [ ]:
# RUN-TOGETHER
import numpy as np

# Same numbers as the failed cell above, now as arrays.
height_m  = np.array([1.73, 1.72, 1.61, 1.89, 1.79])
weight_kg = np.array([65.4, 59.2, 63.6, 88.4, 68.7])

bmi = weight_kg / height_m ** 2
print(bmi)

# The calculation was performed element-wise. Check the first one by hand:
print(65.4 / 1.73 ** 2)

#### **Activity 4.1 - Predict, then run**

Methods are associated with arrays, and may behave differently from methods of the same name on other objects. Lists and arrays react to `+` completely differently. Predict each result before running.

- `[1, 2, 3] + [1, 2, 3]` is going to output ____
- `np.array([1,2,3]) + np.array([1,2,3])` is going to output ____

In [ ]:
# RUN-TOGETHER
python_list = [1, 2, 3]           # standard list
numpy_array = np.array([1, 2, 3]) # standard array

print(python_list + python_list)
print(numpy_array + numpy_array)

print(python_list[1])
print(numpy_array[1])

### 1.3 The homogeneity rule

Because an array holds one type, mixing types forces a promotion to a common type. This is worth seeing on purpose so that it never surprises you.

> **Predict the resulting type before you run the cell.** What happens when a `bool` meets an `int`? When an `int` meets a `str`?

In [ ]:
# RUN-TOGETHER
mixed1 = np.array([True, 1, 2])       # bool + int -> ?
print(mixed1, '->', mixed1.dtype)

mixed2 = np.array([1, 2, 'go'])       # int + str -> ?
print(mixed2, '->', mixed2.dtype)

### 1.4 Broadcasting: array meets scalar

*Broadcasting* is how NumPy applies an operation between an array and a single value: the single value (called a scalar) is applied to every element.

In [ ]:
# RUN-TOGETHER
import seaborn as sns

penguins = sns.load_dataset('penguins')
flippers_mm = penguins['flipper_length_mm'].dropna().to_numpy()

flippers_cm = flippers_mm / 10     # every value divided by 10
print('first five (mm):', flippers_mm[:5])
print('first five (cm):', flippers_cm[:5])

### 1.5 Boolean masks: filtering without loops

Comparing an array to a value gives an array of `True` and `False`. Indexing an array *with* that boolean array keeps only the `True` positions. This replaces filtering loops and reads almost like English.

In [ ]:
# RUN-TOGETHER
print(bmi)

# Suppose we want all of the BMI values over 24.
print(bmi > 24)                        # an array of booleans

print('how many over 24:', sum(bmi > 24))   # False = 0, True = 1

# Return only the values above 24, using the boolean array
print(bmi[bmi > 24])

#### **Activity 4.2 - Heavy penguins**

What fraction of penguins weigh more than 5 kg? Fill in the blanks below.

In [ ]:
# RUN-TOGETHER  (setup for Activity 4.2)
import seaborn as sns, numpy as np

penguins = sns.load_dataset('penguins')
mass_g = penguins['body_mass_g'].dropna().to_numpy()
print('penguins with a recorded mass:', mass_g.size)

In [ ]:
# FILL-IN
# (1) Convert grams to kilograms (1000 g = 1 kg) by broadcasting.
mass_kg = mass_g / ____
print('first five (kg):', mass_kg[:5])

# (2) How many penguins weigh more than 5 kg?
heavy = mass_kg[____]
print('count over 5 kg:', len(heavy))

# (3) What fraction of all penguins is that?
print('fraction:', round(len(heavy) / ____, 3))

#### **Activity 4.3 - Long flippers**

What fraction of penguins have flippers longer than 210 mm?

Note that `flippers_mm` was created back in Section 1.4, and that its length is not the same as `mass_g`, because different penguins are missing different measurements.

In [ ]:
# FILL-IN
# Total number of penguins with a recorded flipper length
total = ____

# Mask out the flippers longer than 210 mm
long = flippers_mm[____]

# Number of penguins with long flippers
n_long = ____

print('fraction of all penguins:', round(n_long / total, 3))

### 1.6 NumPy arrays as objects

Look at the output `<class 'numpy.ndarray'>`:

- the `numpy.` part indicates the object was defined in the numpy package
- the `ndarray` part stands for "n dimensional array"

In [ ]:
# RUN-TOGETHER
import numpy as np
import pandas as pd

# Heights of baseball players, in inches (source: stat.ucla.edu)
heights = pd.read_csv('https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/baseball_heights.csv')
height_in = heights['height_in'].tolist()

print('type of height_in     :', type(height_in))

height_in_array = np.array(height_in)
print('type of the array     :', type(height_in_array))
print('number of players     :', height_in_array.size)

type of height_in     : <class 'list'>
type of the array     : <class 'numpy.ndarray'>
number of players     : 1015


**Definition.**
- `players.shape` is an *attribute*: it reports stored information and uses no parentheses.
- `players.mean()` is a *method*: it performs an action and uses parentheses.

In [ ]:
# RUN-TOGETHER
print('a function approach:', len(height_in_array))
print('using an attribute :', height_in_array.size)

# versus an action
print('using a method     :', height_in_array.mean())

# methods can be stacked!
print('stacking methods   :', height_in_array.mean().round(2))

---
## 2. 2D Arrays, Statistics, and Correlation
---
### 2.1 Two-dimensional arrays

Real data are often organized in a table: many rows, several columns. NumPy represents that as a **2D array**. Now `shape` matters, and indexing takes a `[row, column]` pair.

In [ ]:
# RUN-TOGETHER
import numpy as np

# rows = [heights (in)], [weights (lb)]
players = np.array([[74, 74, 72, 72, 73, 69, 69, 71],          # heights
                    [180, 215, 210, 210, 188, 176, 209, 200]])  # weights

# size does not tell us the dimensions of an n-dimensional array
print('size of the array :', players.size)

# instead we use the shape attribute
print('shape (rows, cols):', players.shape)

In data science we commonly want our data arranged differently from `players`. We prefer each row to be an observational unit (a player) and each column to be a feature (height, weight).

We need to swap the rows and columns so the height and weight *rows* become height and weight *columns*. This is done with the transpose attribute.

In [ ]:
# RUN-TOGETHER
print('original:\n', players)     # \n forces a line break
players_dat = players.T
print('transposed:\n', players_dat)

### 2.2 Indexing and slicing of 2D arrays

> `array_name[row, column]` produces the element in location (row, column)

In Python:
- indexing starts at 0
    - `array_name[0, 0]` is the element in the first row and first column
    - `array_name[-1, -1]` is the last element (negative counts from the end)
- `:` produces all values
- `array_name[row]` or `array_name[row, :]` produces the single row indexed by `row`
- `array_name[:, column]` produces the single column indexed by `column`

We can slice values from rows, from columns, or both, using the notation `start:stop`.
- Remember: the slice runs from `start`, up to but **not including** `stop`.

In [ ]:
# RUN-TOGETHER
print(players_dat)

# data for the first player
print('player 0:', players_dat[0])

# the element in the third row and first column
print(players_dat[2, 0])   # remember 0 indexing!
print(players_dat[2][0])   # also works

# the fourth and fifth players
# Think first:
#   start = 3: the fourth person is in row 3 because indexing starts at 0
#   stop  = 5: we need one more than where we want to stop
print('fourth and fifth:\n', players_dat[3:5, :])

# boolean mask: heights under 70
# Think first:
#   the column of heights is players_dat[:, 0]
#   the boolean mask is players_dat[:, 0] < 70  -> False False ...
print('height < 70:\n', players_dat[players_dat[:, 0] < 70, :])

### 2.3 The `axis` argument

**Definition.** For a 2D array, passing `axis=0` to a method collapses *down the rows*, giving one answer per column. Passing `axis=1` collapses *across the columns*, giving one answer per row.

For example, if we have columns of quantitative features and we want the mean of each feature, we use `array.mean(axis=0)`.

In [ ]:
# RUN-TOGETHER
print(players_dat)
print('mean of each feature (axis=0):\n', players_dat.mean(axis=0))

# What does this one do? Predict before uncommenting.
# print('mean across each row (axis=1):\n', players_dat.mean(axis=1))

---
#### **Activity 4.4 - Penguins, NumPy, and a question of size**

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/02-data_types/palmer_penguins.png?raw=true" width="400">

##### **The scientific story**

The penguin data come from Palmer Station, Antarctica, and were collected to study whether male and female penguins differ in size (*sexual dimorphism*) across three species (Adelie, Chinstrap, Gentoo). In this activity we explore three questions:

- **Part A.** Do males and females differ in size? (review: boolean masks)
- **Part B.** Organize the measurements into a table and summarize them. (review: 2D arrays)
- **Part C.** How do two measurements move together? (bridge to statistics)

> Source: Gorman KB, Williams TD, Fraser WR (2014). *Ecological Sexual Dimorphism and Environmental Variability within a Community of Antarctic Penguins (Genus Pygoscelis).* PLoS ONE 9(3): e90081. Data via the `palmerpenguins` project (Horst, Hill & Gorman).

**Predict every output before you run it.**

In [5]:
# RUN-TOGETHER  (setup for Activity 4.4)
import seaborn as sns
import numpy as np

penguins = sns.load_dataset('penguins').dropna()   # drop rows with any missing value

mass_g   = penguins['body_mass_g'].to_numpy()
flipper  = penguins['flipper_length_mm'].to_numpy()
bill_len = penguins['bill_length_mm'].to_numpy()
bill_dep = penguins['bill_depth_mm'].to_numpy()
sex      = penguins['sex'].to_numpy()      # string array: 'Male' / 'Female'
species  = penguins['species'].to_numpy()  # 'Adelie' / 'Chinstrap' / 'Gentoo'

print('sample size:', len(mass_g))
print('sex values:', np.unique(sex), '| species:', np.unique(species))

sample size: 333
sex values: ['Female' 'Male'] | species: ['Adelie' 'Chinstrap' 'Gentoo']


##### **Part A. Are males bigger than females?**

This is the dimorphism question the data were collected to answer.

**Two new ideas.**

- Combine conditions with `&` (the array version of `and`), and *wrap each condition in parentheses*.
- The *mean of a boolean array is a proportion* (`True` = 1, `False` = 0), so `(mass_g > 4500).mean()` is the fraction of penguins over 4500 g.

In [ ]:
# FILL-IN  (Activity 4.4, Part A)
male   = sex == 'Male'
female = ____                       # Q0. mask for females

# Q1. Compare average mass and flipper length for each sex.
print('mean mass    M / F:', round(mass_g[male].mean(), 1),  '/', round(mass_g[____].mean(), 1))
print('mean flipper M / F:', round(flipper[male].mean(), 2), '/', round(flipper[female].mean(), 2))

# Q2. Count males that are BOTH heavy (> 4500 g) AND long-flippered (> 210 mm).
mask = (sex == 'Male') & (____) & (____)
print('heavy long-flippered males:', mask.sum())

# Q3. What FRACTION of males weigh over 4500 g? And of females?
print('frac of males   over 4500 g:', round((mass_g[male] > 4500).mean(), 3))
print('frac of females over 4500 g:', round((mass_g[____] > 4500).mean(), 3))

# Q4. Flip the direction: AMONG penguins over 4500 g, what fraction are male?
#     Mask to the heavy penguins first, then ask about their sex.
heavy = mass_g > 4500
print('frac of heavy penguins that are male:', round((sex[heavy] == 'Male').mean(), 3))

> **Q5.** Questions 3 and 4 look similar and give different answers. Say in one sentence what each one is actually asking. Which one would you use to answer "are males bigger than females"?

##### **Part B. Put the measurements in a table**

Real datasets store one row per subject and one column per feature. Let's build that layout for flipper length and mass, then summarize it.

In [ ]:
# FILL-IN  (Activity 4.4, Part B)
# Build features-as-rows (2 x n), then transpose to subjects-as-rows.
M = np.array([flipper, mass_g])      # shape (2, n)
D = M.____                           # Q1. transpose -> (n, 2): col 0 flipper, col 1 mass
print('M.shape =', M.shape, '   D.shape =', D.shape)

# Q2. Mean, min, and max of EACH feature (a flipper answer and a mass answer).
#     Which axis collapses down the rows to give one number per column?
print('feature means:', np.round(D.mean(axis=____), 2))
print('feature mins :', D.min(axis=____))
print('feature maxs :', D.max(axis=____))

# Q3. Broadcasting: convert the mass column (col 1) from grams to kilograms.
mass_kg = D[:, 1] / ____
print('first three masses (kg):', mass_kg[:3])

# Q4. Fancy indexing: find the row of the HEAVIEST penguin and report its flipper length.
i = np.argmax(D[:, 1])      # position of the largest mass
print('heaviest penguin row:', D[i], '  its flipper (mm):', D[i, ____])

# Q5. Combine Parts A and B: mean flipper length of penguins over 4500 g,
#     using a boolean mask on the ROWS of D.
row_mask = D[:, 1] > 4500
print('mean flipper of heavy penguins (mm):', round(D[row_mask, ____].mean(), 2))

##### **Part C. When one number lies**

Bigger penguins should have bigger flippers. We measure how tightly two variables move together with a **correlation**. `np.corrcoef` returns a small matrix whose off-diagonal entry `[0, 1]` is the correlation of the two inputs.

- correlation values lie between -1 and 1
- negative values imply a negative linear association between the two variables
- positive values imply a positive linear association
- correlations closer to -1 or 1 suggest strong associations; values close to 0 imply no linear association

Compute a correlation for all penguins pooled, then within each species.

In [ ]:
# FILL-IN  (Activity 4.4, Part C)
# Three species masks. We reuse them for both comparisons below.
adelie    = species == 'Adelie'
chinstrap = species == 'Chinstrap'
gentoo    = species == '____'      # Q1. fill in the last species name

# Q2. Flipper length vs body mass. The off-diagonal entry [0, 1] is the correlation.
print('FLIPPER vs MASS')
print('  pooled    r =', round(np.corrcoef(flipper, mass_g)[0, ____], 3))   # fill in the index
print('  Adelie    r =', round(np.corrcoef(flipper[adelie],    mass_g[adelie])[0, 1], 3))
print('  Chinstrap r =', round(np.corrcoef(flipper[chinstrap], mass_g[chinstrap])[0, 1], 3))
print('  Gentoo    r =', round(np.corrcoef(flipper[gentoo],    mass_g[gentoo])[0, 1], 3))

FLIPPER vs MASS
  pooled    r = 0.873
  Adelie    r = 0.465
  Chinstrap r = 0.642
  Gentoo    r = 0.711


In [9]:
# RUN-TOGETHER  (Activity 4.4, Part C continued)
# Q3. Bill length vs bill depth. PREDICT THE SIGN of the pooled correlation first.
print('BILL LENGTH vs BILL DEPTH')
print('  pooled    r =', round(np.corrcoef(bill_len, bill_dep)[0, 1], 3))
print('  Adelie    r =', round(np.corrcoef(bill_len[adelie],    bill_dep[adelie])[0, 1], 3))
print('  Chinstrap r =', round(np.corrcoef(bill_len[chinstrap], bill_dep[chinstrap])[0, 1], 3))
print('  Gentoo    r =', round(np.corrcoef(bill_len[gentoo],    bill_dep[gentoo])[0, 1], 3))

BILL LENGTH vs BILL DEPTH
  pooled    r = -0.229
  Adelie    r = 0.386
  Chinstrap r = 0.654
  Gentoo    r = 0.654


> **Q4. Discuss.** For bill length versus bill depth, is the honest relationship positive or negative? Name the lurking variable. Why does pooling the species reverse the story?

**This phenomenon has a name: Simpson's paradox.** A trend that holds within every group can disappear, or reverse, when the groups are combined. Here, each species has a positive bill length versus depth relationship, but Gentoo penguins have long, shallow bills while Adelies have short, deep ones. Pooling them lets the *between-species* difference overwhelm the *within-species* pattern, and the pooled correlation comes out negative.

The practical lesson is not that pooling is always wrong. It is that a single correlation computed over a mixed population can point the opposite direction from the truth about every individual in it, and nothing in the number itself warns you. You will see this again when we study confounding in the EXPLORE unit.

### 2.4 Basic statistics

What else can we do with this data? Useful NumPy functions and methods include:

- `mean()`
- `median()`
- `corrcoef()`
- `std()`
- `sum()`
- `sort()`

#### **Activity 4.5 - z-scores and outliers**

> **Small-group investigation.** A common data science measurement is the **z-score**:
>
> z = (value - mean) / standard deviation
>
> which re-expresses each value as "how many standard deviations from the mean."

Use it to flag unusual penguins. Fill in the blanks, and predict before running.

In [ ]:
# RUN-TOGETHER  (setup for Activity 4.5)
import seaborn as sns, numpy as np

penguins = sns.load_dataset('penguins')
mass = penguins['body_mass_g'].dropna().to_numpy()
print('n =', mass.size, '| mean =', round(mass.mean(), 1), '| std =', round(mass.std(), 1))

In [ ]:
# FILL-IN
# (1) Standardize the masses to z-scores.
z = (mass - mass.mean()) / ____

# (2) By construction, z should have mean ~0 and std ~1. Check:
print('z mean:', round(z.mean(), 10))
print('z std :', round(z.std(), 10))

# (3) Flag penguins more than 2 SD from the mean (unusually big or small).
unusual = mass[np.abs(z) ____ 2]
print('number of unusual penguins:', unusual.size)
print('their masses:', np.sort(unusual))

> **Discuss.** A z-score flags a value as unusual *relative to the whole sample*. Given what you found in Activity 4.4 Part C, what is the risk of applying this to all three penguin species pooled together? Would you expect a Gentoo to be flagged more or less often than it should be?

## Suggested Exercises

1. Consider these two objects:

    ```python
    L = [2, 4, 6, 8]
    A = np.array([2, 4, 6, 8])
    ```

    a. Predict the output of `L * 2` and of `A * 2`. Explain why they differ.

    b. Predict the output of `L + [1]` and of `A + 1`. One of these raises an error. Which, and why?

    c. Write the one-line NumPy expression that gives the square of every element of `A`.

    d. Explain in one or two sentences why NumPy's behavior is the one a data scientist usually wants.

2. An array holds daily rainfall in millimeters for two weeks:

    ```python
    rain = np.array([0, 0, 3.2, 11.4, 0, 0.6, 0, 0, 0, 22.1, 4.5, 0, 0, 1.1])
    ```

    a. Write a boolean mask for days with any measurable rain.

    b. Using that mask, compute how many days had rain and what fraction of the two weeks that represents.

    c. Compute the mean rainfall across all 14 days, and the mean across only the rainy days. Which is larger, and which better answers "how hard does it rain here when it rains"?

    d. A colleague reports "average daily rainfall was 3.06 mm" without saying which calculation was used. Explain why that sentence is not yet information in the Module 2 sense.

3. A 2D array holds four students, with columns for exam 1, exam 2, and the final:

    ```python
    scores = np.array([[88, 92, 85],
                       [79, 71, 90],
                       [95, 98, 99],
                       [62, 70, 74]])
    ```

    a. What does `scores.shape` return? Which number is the number of students?

    b. Write the expression for the class mean on each exam. Which axis, and why?

    c. Write the expression for each student's mean across their three scores.

    d. Write a boolean mask selecting students whose final exceeded 80, and use it to report those students' exam 1 scores.

    e. `scores.mean()` with no axis returns a single number. Say what that number means, and give one reason it might be misleading to report it.

4. A researcher studies whether hours studied predicts exam score, using data from two courses pooled together. She finds a correlation near zero and concludes studying does not help.

    a. Describe how the within-course correlations could both be strongly positive while the pooled correlation is near zero. A sketch in words is fine.

    b. Name the phenomenon, and name the lurking variable in this scenario.

    c. What single additional column would let you check her conclusion?

    d. Connect this to Activity 4.4 Part C: what did pooling do to the penguin bill measurements, and why is "compute one correlation over everything" a risky default?